In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [4]:
df = pd.read_csv("qoute_dataset.csv")
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [5]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [6]:
df.shape

(3038, 2)

In [7]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [8]:
quotes = quotes.str.lower()

In [9]:
import string
translator = str.maketrans('','', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [10]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [12]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [13]:
word_index= tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [14]:
sequence = tokenizer.texts_to_sequences(quotes)

In [15]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [16]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [17]:
x = []
y = []

for seq in sequence:
    for i in range (1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        x.append(input_seq)
        y.append(output_seq)


In [18]:
len(x)

85271

In [19]:
len(y)

85271

In [20]:
max_len = max(len(x) for x in x)
print(max_len)

745


In [21]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
x_padded = pad_sequences(x, maxlen=max_len, padding='pre')


In [22]:
x_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]], dtype=int32)

In [23]:
y= np.array(y)

In [24]:
x_padded.shape

(85271, 745)

In [25]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y,num_classes=vocab_size)


In [26]:
y.shape

(85271,)

In [27]:
y_one_hot.shape

(85271, 10000)

In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense


In [29]:
embedding_dim = 50
rnn_units = 128


In [30]:
rnn_model = Sequential ([
    Embedding (input_dim=vocab_size, output_dim=embedding_dim,
    input_length=max_len),

])
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [31]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossemtropy',
    metrics=['accuracy']
)

In [32]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [33]:
lstm_model= Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim,
              input_length=max_len)
)

lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [34]:
rnn_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [35]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [36]:
epochs=60
batch_size = 128

In [37]:
history_rnn = rnn_model.fit(
    x_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/60
600/600 ━━━━━━━━━━━━━━━━━━━━ 46s 68ms/step - accuracy: 0.0449 - loss: 6.7103 - val_accuracy: 0.0558 - val_loss: 6.5485
Epoch 2/60
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 61ms/step - accuracy: 0.0801 - loss: 6.0988 - val_accuracy: 0.0896 - val_loss: 6.3324
Epoch 3/60
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 62ms/step - accuracy: 0.1033 - loss: 5.7349 - val_accuracy: 0.1026 - val_loss: 6.2922
Epoch 4/60
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 62ms/step - accuracy: 0.1180 - loss: 5.4442 - val_accuracy: 0.1044 - val_loss: 6.3214
Epoch 5/60
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 61ms/step - accuracy: 0.1305 - loss: 5.1883 - val_accuracy: 0.1075 - val_loss: 6.3809
Epoch 6/60
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 62ms/step - accuracy: 0.1429 - loss: 4.9514 - val_accuracy: 0.1088 - val_loss: 6.4300
Epoch 7/60
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 61ms/step - accuracy: 0.1570 - loss: 4.7306 - val_accuracy: 0.1085 - val_loss: 6.5079
Epoch 8/60
600/600 ━━━━━━━━━━━━━━━━━━━━ 37s 61ms/step - accuracy: 0.1759 - loss: 4.5237 - 

In [38]:
lstm_model.save("next_word_prediction.h5")

In [39]:
index_to_word = {}

for word, index in word_index.items():

    index_to_word[index] = word

In [40]:
from tensorflow.keras.models import load_model

model = load_model("next_word_prediction.h5", compile=False)

In [41]:
def predict_next_word(model,tokenizer,text,max_len):

    text = text.lower()

    seq = tokenizer.texts_to_sequences([text])[0]

    seq = pad_sequences([seq], maxlen=max_len, padding='pre')

    preds = model.predict(seq, verbose=0)
    pred_index = np.argmax(preds)
    return index_to_word[pred_index]

In [42]:
seed_text = "what are you"
next_word = predict_next_word(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

feet


In [43]:

# =====================================
# 13. Generate Sentence
# =====================================
def generate_text(model,tokenizer,seed_text,max_len, n_words):

    for i in range(n_words):

        next_word = predict_next_word(model,tokenizer,seed_text,max_len)

        if next_word == "":
            break

        seed_text += " " + next_word

    return seed_text

In [47]:
print(len(index_to_word))

8978


In [55]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_next_word(model, tokenizer, text, max_len):
    # text → sequence
    seq = tokenizer.texts_to_sequences([text])[0]

    # padding
    seq = pad_sequences([seq], maxlen=max_len-1, padding='pre')

    # prediction
    preds = model.predict(seq, verbose=0)
    pred_index = preds.argmax()

    # reverse mapping
    index_to_word = {index: word for word, index in tokenizer.word_index.items()}

    return pred_index, index_to_word.get(pred_index, "<UNK>")

In [ ]:
text = "what are you"

pred_index, word = predict_next_word(lstm_model, tokenizer, text, max_len)
print(pred_index)
print(word)

In [57]:

import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [58]:
with open("max_len.pkl", "wb") as f:
    pickle.dump(max_len, f)